# RTU 전력 예측 모델링
### 피처 전략
- current/reactivePower → 5월 데이터 없어서 사용 불가
- 사용 가능 피처: hour_sin/cos + lag (과거값)
- 예측 방식: 재귀 예측 (1시간씩 순서대로)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'  # Mac: 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
print('라이브러리 로드 완료')

## 1. 데이터 로드 & hourly_pow 생성

In [ ]:
# train.csv 로드
df = pd.read_csv('train.csv')
df['dt'] = pd.to_datetime(df['localtime'], format='%Y%m%d%H%M%S')

# 설비별 1시간 평균 → 전체 합산
hourly = (
    df.groupby(['module(equipment)', pd.Grouper(key='dt', freq='1H')])
    ['activePower'].mean()
    .reset_index()
    .groupby('dt')['activePower'].sum()
    .reset_index()
    .rename(columns={'activePower': 'hourly_pow'})
)

print('hourly shape:', hourly.shape)
print('기간:', hourly['dt'].min(), '~', hourly['dt'].max())
print(hourly.head())

## 2. 피처 생성
> hour_sin/cos + lag + rolling

In [ ]:
# 시간 피처
hourly['hour'] = hourly['dt'].dt.hour
hourly['hour_sin'] = np.sin(2 * np.pi * hourly['hour'] / 24)
hourly['hour_cos'] = np.cos(2 * np.pi * hourly['hour'] / 24)

# lag 피처 (과거값)
hourly['lag_1']   = hourly['hourly_pow'].shift(1)
hourly['lag_24']  = hourly['hourly_pow'].shift(24)
hourly['lag_168'] = hourly['hourly_pow'].shift(168)

# rolling 피처 (직전 n시간 통계)
hourly['rolling_mean_24']  = hourly['hourly_pow'].shift(1).rolling(24).mean()
hourly['rolling_std_24']   = hourly['hourly_pow'].shift(1).rolling(24).std()
hourly['rolling_mean_168'] = hourly['hourly_pow'].shift(1).rolling(168).mean()

# 결측치 제거 (lag 생성시 앞부분 NaN)
hourly = hourly.dropna().reset_index(drop=True)

print('피처 생성 완료')
print('shape:', hourly.shape)
print(hourly.head())

## 3. 학습/검증 분리
> 시간 기준 분할 — 12월~3월 학습 / 4월 검증
> (5월은 예측 대상이라 정답 없음)

In [ ]:
# 12월~3월 학습 / 4월 검증
train = hourly[hourly['dt'] < '2025-04-01'].copy()
valid = hourly[hourly['dt'] >= '2025-04-01'].copy()

features = [
    'hour', 'hour_sin', 'hour_cos',
    'lag_1', 'lag_24', 'lag_168',
    'rolling_mean_24', 'rolling_std_24', 'rolling_mean_168'
]
target = 'hourly_pow'

X_train = train[features]
y_train = train[target]
X_valid = valid[features]
y_valid = valid[target]

print(f'학습: {train.shape} ({train["dt"].min().date()} ~ {train["dt"].max().date()})')
print(f'검증: {valid.shape} ({valid["dt"].min().date()} ~ {valid["dt"].max().date()})')

## 4. LightGBM 학습

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    verbose=-1
)
model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# 4월 검증 성능
y_pred_valid = model.predict(X_valid)
mae   = mean_absolute_error(y_valid, y_pred_valid)
rmse  = np.sqrt(mean_squared_error(y_valid, y_pred_valid))
smape = np.mean(2 * np.abs(y_pred_valid - y_valid) / (np.abs(y_pred_valid) + np.abs(y_valid))) * 100

print(f'[4월 검증 성능]')
print(f'  MAE:   {mae:.2f}')
print(f'  RMSE:  {rmse:.2f}')
print(f'  SMAPE: {smape:.4f}%')

In [ ]:
# 피처 중요도
plt.figure(figsize=(10, 5))
lgb.plot_importance(model, max_num_features=10, importance_type='gain')
plt.title('LightGBM 피처 중요도')
plt.tight_layout()
plt.savefig('model_01_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

# 4월 예측 시각화
plt.figure(figsize=(18, 5))
plt.plot(valid['dt'], y_valid.values, label='실제값', color='steelblue', linewidth=0.8)
plt.plot(valid['dt'], y_pred_valid, label='예측값', color='tomato', linewidth=0.8, alpha=0.8)
plt.title('LightGBM 4월 검증 결과')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_02_valid_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 5월 재귀 예측
> 1시간씩 순서대로 예측 → 예측값을 lag에 반영 → 다음 시간 예측

In [ ]:
# 5월 예측 시간대 생성 (672시간)
may_dates = pd.date_range('2025-05-01', periods=672, freq='1H')

# 과거 실제값 히스토리 (lag 계산용)
history = hourly[['dt', 'hourly_pow']].copy()

may_preds = []

for dt in may_dates:
    hour = dt.hour
    hour_sin = np.sin(2 * np.pi * hour / 24)
    hour_cos = np.cos(2 * np.pi * hour / 24)

    # lag 값 계산
    def get_val(h):
        row = history[history['dt'] == dt - pd.Timedelta(hours=h)]
        return row['hourly_pow'].values[0] if len(row) > 0 else np.nan

    lag_1   = get_val(1)
    lag_24  = get_val(24)
    lag_168 = get_val(168)

    # rolling 계산
    recent_24  = history.tail(24)['hourly_pow']
    recent_168 = history.tail(168)['hourly_pow']
    rolling_mean_24  = recent_24.mean()
    rolling_std_24   = recent_24.std()
    rolling_mean_168 = recent_168.mean()

    X_pred = pd.DataFrame([[
        hour, hour_sin, hour_cos,
        lag_1, lag_24, lag_168,
        rolling_mean_24, rolling_std_24, rolling_mean_168
    ]], columns=features)

    pred = model.predict(X_pred)[0]
    may_preds.append(pred)

    # 예측값을 히스토리에 추가 (다음 예측의 lag로 사용)
    new_row = pd.DataFrame({'dt': [dt], 'hourly_pow': [pred]})
    history = pd.concat([history, new_row], ignore_index=True)

print(f'5월 예측 완료: {len(may_preds)}시간')
print(f'예측값 범위: {min(may_preds):.2f} ~ {max(may_preds):.2f}')

In [ ]:
# 5월 예측 시각화
plt.figure(figsize=(18, 5))
plt.plot(may_dates, may_preds, color='tomato', linewidth=0.8, label='5월 예측값')
plt.title('5월 hourly_pow 재귀 예측 결과')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_03_may_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 이상탐지 — Rolling Z-score

In [ ]:
hourly['rolling_mean'] = hourly['hourly_pow'].rolling(window=24, min_periods=1).mean()
hourly['rolling_std2'] = hourly['hourly_pow'].rolling(window=24, min_periods=1).std()
hourly['z_score'] = (
    (hourly['hourly_pow'] - hourly['rolling_mean']) / hourly['rolling_std2']
)

threshold = 2.5
hourly['is_anomaly'] = hourly['z_score'].abs() > threshold
anomalies = hourly[hourly['is_anomaly']]

print(f'이상 탐지: {len(anomalies)}건 / 전체 {len(hourly)}시간')

plt.figure(figsize=(18, 5))
plt.plot(hourly['dt'], hourly['hourly_pow'],
         color='steelblue', linewidth=0.6, label='hourly_pow')
plt.scatter(anomalies['dt'], anomalies['hourly_pow'],
            color='red', s=20, zorder=5, label=f'이상탐지 (Z>{threshold})')
plt.title('Rolling Z-score 이상탐지')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_04_anomaly_zscore.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. 이상탐지 — Isolation Forest

In [ ]:
from sklearn.ensemble import IsolationForest

X_iso = hourly[['hourly_pow', 'lag_1', 'lag_24', 'rolling_mean_24']].dropna()

iso_model = IsolationForest(contamination=0.01, random_state=42, n_estimators=100)
hourly.loc[X_iso.index, 'iso_pred'] = iso_model.fit_predict(X_iso)

iso_anomalies = hourly[hourly['iso_pred'] == -1]
print(f'Isolation Forest 이상 탐지: {len(iso_anomalies)}건')

plt.figure(figsize=(18, 5))
plt.plot(hourly['dt'], hourly['hourly_pow'],
         color='steelblue', linewidth=0.6, label='hourly_pow')
plt.scatter(iso_anomalies['dt'], iso_anomalies['hourly_pow'],
            color='orange', s=20, zorder=5, label='Isolation Forest 이상')
plt.title('Isolation Forest 이상탐지')
plt.xlabel('날짜')
plt.ylabel('hourly_pow (kW)')
plt.legend()
plt.tight_layout()
plt.savefig('model_05_anomaly_iforest.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 제출 파일 생성

In [ ]:
# agg_pow = 5월 전체 누적 합계
agg_pow    = sum(may_preds)
may_bill   = agg_pow * 180
may_carbon = agg_pow * 0.424

print(f'agg_pow:    {agg_pow:,.2f} kWh')
print(f'may_bill:   {may_bill:,.0f} 원')
print(f'may_carbon: {may_carbon:,.2f} kgCO₂')

submission = pd.DataFrame({
    'id':         may_dates.strftime('%Y-%m-%d %H:%M:%S'),
    'hourly_pow': may_preds,
    'agg_pow':    agg_pow,    # 전 행 동일값
    'may_bill':   may_bill,
    'may_carbon': may_carbon
})

submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv 저장 완료!')
print(submission.head(10))